In [1]:
TRAINING_FILE = "code_files/funding_train.jsonl"
VALIDATION_FILE = "code_files/funding_val.jsonl"

In [2]:
import json

def validate_jsonl(path):
    with open(path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            print(f"[Error] Line {i+1}: {e}")

    print(f"✅ Checked {len(lines)} lines in {path}")

validate_jsonl(TRAINING_FILE)
validate_jsonl(VALIDATION_FILE)

✅ Checked 281 lines in code_files/funding_train.jsonl
✅ Checked 71 lines in code_files/funding_val.jsonl


In [ ]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ACCELERATE_USE_MPS"] = "False"
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": TRAINING_FILE,
    "validation": VALIDATION_FILE
})

def format_instruction(example):
    """Format the prompt and completion as an instruction."""
    text = f"### Instruction:\n{example['prompt']}\n\n### Response:\n{example['completion']}"
    return {"text": text}

formatted_dataset = dataset.map(format_instruction)


Setting up tokenizer...
Tokenizing dataset...
Loading model...


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Applying LoRA...
Setting up trainer...
Starting training...


Step,Training Loss
1,3.192500
2,2.767900
3,3.768500
4,3.280500
5,2.889900
6,3.522400
7,3.826800
8,3.118500
9,2.992600
10,3.859600


Training successful!


In [ ]:
print("Setting up tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # Very short sequences for testing
    )

print("Tokenizing dataset...")
tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

In [ ]:
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="cpu",
    torch_dtype=torch.float32
)

In [ ]:
print("Applying LoRA...")
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Just one module
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./test-output",
    num_train_epochs=4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_steps=1,
    save_steps=10,
    eval_steps=5,
    learning_rate=1e-10,
    weight_decay=0.0,
    fp16=False,
    bf16=False,
    report_to="none",
    use_cpu=True
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Setting up trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

In [ ]:
print("Starting training...")
try:
    trainer.train()
    print("Training successful!")
except Exception as e:
    import traceback
    print(f"Error during training: {e}")
    traceback.print_exc()

In [124]:
from transformers import pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who helps users find information about funding opportunities.",
    },
    {"role": "user", "content": "Are there any special conditions or gotchas when applying to Rufford Foundation?"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

Device set to use cpu
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForConditionalGeneration', 'Gemma3ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'Glm4ForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCa

<|system|>
You are a friendly chatbot who helps users find information about funding opportunities.</s>
<|user|>
Are there any special conditions or gotchas when applying to Rufford Foundation?</s>
<|assistant|>
Yes, there are some special conditions or gotchas when applying to Rufford Foundation. Here are some key points to consider:

1. Eligibility: The Rufford Foundation offers funding opportunities to individuals, schools, and communities. You must meet the eligibility requirements for the program you are applying for.

2. Budget: The Rufford Foundation provides funding up to a maximum of £10,000. You must provide a detailed budget for your project or activity, which must show how the funding will be used.

3. Previous experience: The Rufford Foundation requires applicants to have some previous experience in the field of environment, science, or conservation. You must have experience or knowledge in the specific area of interest for the funding opportunity you are applying for.

4.